In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-2-3-mouse-boxplot-MLP

Plot distributions and comparisons from mouse benchmark results.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


boxplot

In [ ]:
# ============================================================



# ============================================================

import os
import sys
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False

BASE_SIZE = 7
plt.rcParams.update({
    "font.size": BASE_SIZE,
    "axes.labelsize": BASE_SIZE + 1,
    "axes.titlesize": BASE_SIZE + 2,
    "xtick.labelsize": BASE_SIZE,
    "ytick.labelsize": BASE_SIZE,
    "legend.fontsize": BASE_SIZE,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "lines.linewidth": 0.5,
})


input_data_dir = input_path("1-TMS-remove/4-precision-5_to_25/1-5_to_25-pcc-add-model-mlp-tanh-parallel")
full_results_file = os.path.join(input_data_dir, "1-end-mlp_tanh_random_sample_parallel.csv")


target_benchmark = "aging-map.xlsx"

plot_output_dir = output_path("2-8.3-shanda/1-feature/1-figure/0-2-result-3-benchmark/2-1-boxplot-PCC-MAE-mlp-tanh")
os.makedirs(plot_output_dir, exist_ok=True)

try:
    df_full = pd.read_csv(full_results_file)
except FileNotFoundError:
    print(f"Error: {full_results_file} 不存在，请先运行 6-benchmark-mouse-mlp.ipynb 的并行计算代码。")
    sys.exit(1)

for col in ["Precision", "Pearson_R", "MAE", "RMSE"]:
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors="coerce")

required_cols = {"Model", "Benchmark_Name", "Tissue", "Threshold", "Precision", "Pearson_R", "MAE", "RMSE"}
missing_cols = required_cols.difference(df_full.columns)
if missing_cols:
    raise ValueError(f"full results CSV 缺少必要列: {sorted(missing_cols)}")

df_clean = df_full.dropna(subset=["Precision", "Pearson_R", "MAE", "RMSE"], how="all").copy()

models_to_include = {"scimmuaging", "buckley", "scale", "maple", "xgboost", "iage", "elasticnet"}
df_clean = df_clean[
    df_clean["Model"].astype(str).str.lower().isin(models_to_include)
].copy()

if df_clean.empty:
    print("数据为空，请检查原始数据。")
    sys.exit(0)


def format_model_name(model):
    model_str = str(model).strip()
    model_lower = model_str.lower()
    if model_lower == "maple":
        return "Sage"
    if model_lower == "scimmuaging":
        return "sc-ImmuAging"
    if model_lower == "iage":
        return "iAge"
    return model_str[:1].upper() + model_str[1:].lower()

df_clean["Model_Raw"] = df_clean["Model"].astype(str)
df_clean["Model"] = df_clean["Model"].map(format_model_name)

all_benchmarks = df_clean["Benchmark_Name"].dropna().unique().tolist()


target_order = ["Sage", "Xgboost", "Buckley", "Scale", "sc-ImmuAging", "iAge", "Elasticnet"]
raw_models = df_clean["Model"].unique().tolist()
global_model_order = [m for m in target_order if m in raw_models]
for m in raw_models:
    if m not in global_model_order:
        global_model_order.append(m)


def create_global_palette(models, highlight_name="Sage"):
    other_colors = ["#4DBBD5", "#00A087", "#3C5488", "#F39B7F", "#8491B4", "#91D1C2", "#7E6148", "#B09C85"]
    palette = {}
    color_iter = itertools.cycle(other_colors)
    for m in models:
        if highlight_name in m:
            palette[m] = "#E64B35"
        else:
            palette[m] = next(color_iter)
    return palette


global_palette = create_global_palette(global_model_order)


def get_stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def get_clean_bench_name(bench_raw):
    bench_lower = str(bench_raw).lower()
    if "aging" in bench_lower and "map" in bench_lower:
        return "Aging Map"
    if "csgene" in bench_lower:
        return "CSGene"
    if "agingatlas" in bench_lower:
        return "Aging Atlas"
    return str(bench_raw).replace(".xlsx", "").replace(".csv", "")


def get_metric_display_name(metric):
    if metric == "MAE":
        return "MAE(months)"
    if metric == "Pearson_R":
        return "Pearson R"
    return metric


def safe_name(value):
    return str(value).replace("/", "-").replace(" ", "_").replace(".", "_")


plot_tasks = []
for metric in ["Precision", "Pearson_R", "MAE", "RMSE"]:
    if metric not in df_clean.columns:
        continue
    if metric == "Precision":
        for b in all_benchmarks:
            plot_tasks.append((metric, b))
    else:
        if target_benchmark in all_benchmarks:
            plot_tasks.append((metric, target_benchmark))
        else:
            plot_tasks.append((metric, all_benchmarks[0]))


thresholds = sorted(df_clean["Threshold"].dropna().unique())
saved_paths = []

for metric, current_bench in plot_tasks:
    plot_df = df_clean[df_clean["Benchmark_Name"] == current_bench].dropna(subset=[metric]).copy()
    if plot_df.empty:
        continue

    current_plot_models = [m for m in global_model_order if m in plot_df["Model"].unique()]
    plot_df = plot_df[plot_df["Model"].isin(current_plot_models)]
    if not current_plot_models:
        continue


    aligned_slices = []
    for thres in thresholds:
        df_slice = plot_df[plot_df["Threshold"] == thres].copy()
        if df_slice.empty:
            continue
        pivot = df_slice.pivot_table(index="Tissue", columns="Model", values=metric, aggfunc="mean")
        cols_to_check = [m for m in current_plot_models if m in pivot.columns]
        if not cols_to_check:
            continue
        pivot_clean = pivot.dropna(subset=cols_to_check)
        common_tissues = pivot_clean.index.tolist()
        df_slice_aligned = df_slice[df_slice["Tissue"].isin(common_tissues)]
        aligned_slices.append(df_slice_aligned)

    if not aligned_slices:
        continue
    plot_df_aligned = pd.concat(aligned_slices, ignore_index=True)


    ttest_results = []
    sage_model_name = next((m for m in current_plot_models if "Sage" in m), None)

    for thres in thresholds:
        df_slice = plot_df_aligned[plot_df_aligned["Threshold"] == thres]
        if df_slice.empty:
            continue
        pivot = df_slice.pivot_table(index="Tissue", columns="Model", values=metric, aggfunc="mean")
        for i in range(len(current_plot_models)):
            for j in range(i + 1, len(current_plot_models)):
                m1, m2 = current_plot_models[i], current_plot_models[j]
                if m1 in pivot.columns and m2 in pivot.columns:
                    data = pivot[[m1, m2]].dropna()
                    if len(data) >= 3:
                        _, p = stats.ttest_rel(data[m1], data[m2])
                        ttest_results.append({
                            "Threshold": thres,
                            "Model_1": m1,
                            "Model_2": m2,
                            "P_Value": p,
                        })
    df_ttest = pd.DataFrame(ttest_results)

    clean_bench_name = get_clean_bench_name(current_bench)
    metric_label = get_metric_display_name(metric)
    y_label_text = f"{metric_label} ({clean_bench_name})" if metric == "Precision" else metric_label

    for thres in thresholds:
        subset = plot_df_aligned[plot_df_aligned["Threshold"] == thres].copy()
        if subset.empty:
            continue

        fig_width_inch = 85 / 25.4
        fig_height_inch = 73 / 25.4
        fig, ax = plt.subplots(figsize=(fig_width_inch, fig_height_inch))

        ax.grid(True, axis="both", color="#EAEAEA", linestyle="-", linewidth=0.5, zorder=0)

        sns.boxplot(
            data=subset,
            x="Model",
            y=metric,
            order=current_plot_models,
            ax=ax,
            palette=global_palette,
            hue="Model",
            legend=False,
            width=0.55,
            linewidth=0.5,
            fliersize=0,
            boxprops=dict(alpha=0.9, edgecolor="black"),
            zorder=2,
        )

        sns.stripplot(
            data=subset,
            x="Model",
            y=metric,
            order=current_plot_models,
            ax=ax,
            color="#404040",
            alpha=0.6,
            size=2.0,
            jitter=0.2,
            zorder=2,
        )

        y_min, y_max = ax.get_ylim()
        current_y_top = subset[metric].max()
        if pd.isna(current_y_top):
            current_y_top = y_max

        anno_list = []
        if not df_ttest.empty and sage_model_name:
            sig_subset = df_ttest[
                (df_ttest["Threshold"] == thres)
                & (df_ttest["P_Value"] < 0.05)
                & ((df_ttest["Model_1"] == sage_model_name) | (df_ttest["Model_2"] == sage_model_name))
            ]
            for _, row in sig_subset.iterrows():
                m1, m2 = row["Model_1"], row["Model_2"]
                if m1 not in current_plot_models or m2 not in current_plot_models:
                    continue
                if "sage" in m2.lower():
                    m1, m2 = m2, m1
                if "sage" in m1.lower():
                    x1 = current_plot_models.index(m1)
                    x2 = current_plot_models.index(m2)
                    anno_list.append((x1, x2, row["P_Value"], abs(x2 - x1)))
            anno_list.sort(key=lambda x: x[3])

        total_lines = len(anno_list)
        if metric == "Pearson_R":
            available_space = 1.0 - current_y_top
            if total_lines > 0:
                step = available_space / (total_lines * 1.2 + 1.0)
                if step < 0.015:
                    step = 0.015
            else:
                step = 0.05
        else:
            step = (y_max - y_min) * 0.055
            if step <= 0 or not np.isfinite(step):
                step = 0.05

        annotation_count = 0
        for x1, x2, p_val, _ in anno_list:
            h = current_y_top + step * (annotation_count * 1.2 + 1.2)
            tick_len = step * 0.35
            ax.plot([x1, x1, x2, x2], [h - tick_len, h, h, h - tick_len], lw=0.5, c="black", zorder=3)
            ax.text(
                (x1 + x2) / 2,
                h + step * 0.05,
                get_stars(p_val),
                ha="center",
                va="bottom",
                fontsize=BASE_SIZE,
                color="black",
                zorder=3,
            )
            annotation_count += 1

        ax.set_title(f"Threshold: {thres}", fontweight="bold", pad=8)
        ax.set_xlabel("")
        ax.set_ylabel(y_label_text, fontweight="bold")
        ax.set_xticklabels(current_plot_models, rotation=45, ha="right", rotation_mode="anchor")
        ax.tick_params(axis="x", pad=1)
        sns.despine(ax=ax)

        if metric == "Pearson_R":
            lower = min(y_min, subset[metric].min() - 0.03)
            ax.set_ylim(lower, 1.0)
        else:
            required_y_max = current_y_top + step * (annotation_count * 1.2 + 2.5) if annotation_count > 0 else current_y_top + step * 2
            ax.set_ylim(y_min, required_y_max)

        plt.tight_layout()
        bench_suffix = safe_name(str(current_bench).replace(".xlsx", "").replace(".csv", ""))
        save_name = f"MLP_Tanh_Figure_{metric}_{bench_suffix}_Thres{thres}_StaircaseStyle.pdf"
        save_path = os.path.join(plot_output_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches="tight", transparent=True)
        plt.close(fig)
        saved_paths.append(save_path)
        print(f"[{metric} - {current_bench} - Thres:{thres}] mlp_tanh 阶梯标注版已保存: {save_path}")

pd.DataFrame({"figure_path": saved_paths}).to_csv(
    os.path.join(plot_output_dir, "MLP_Tanh_Boxplot_Figure_Index.csv"),
    index=False,
)

print(f"\n完成。共保存 {len(saved_paths)} 张 PDF 箱线图。")
print(f"输出目录: {plot_output_dir}")
